In [ ]:
import pandas as pd 
import numpy as np 

import matplotlib.pyplot as plt 
import matplotlib.gridspec as gridspec

import seaborn as sns 

import gower
from sklearn.cluster import DBSCAN

import statsmodels.formula.api as smf 
from scipy.stats import spearmanr, sem, pearsonr, norm, chi2

from copy import deepcopy

import matplotlib
import matplotlib.pyplot as plt 
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import MaxNLocator, PercentFormatter

from sklearn.metrics import mean_squared_error


# Utility functions and data processing 

In [ ]:
# Load dataframes 
directory = "./data/"
df_paired_val = pd.read_csv(directory + "df_paired_val.csv")
df_paired_learn = pd.read_csv(directory + "df_paired_learn.csv")
df_analysis_val = pd.read_csv(directory + "df_analysis_val.csv")
df_analysis_learn = pd.read_csv(directory + "df_analysis_learn.csv")
df_rounds_learn = pd.read_csv(directory + "df_rounds_learn.csv")
df_rounds_val = pd.read_csv(directory + "df_rounds_val.csv")
df_predictions = pd.read_csv(directory + "df_predictions.csv")

In [ ]:
ate_estimate = smf.ols("itt_efficiency ~ CONFIG_punishmentExists", data=df_analysis_learn.query("valid_number_of_starting_players")).fit().params["CONFIG_punishmentExists[T.True]"]
df_paired_val["baseline"] = df_paired_learn["treatment_itt_efficiency"].mean()
df_paired_val["no_effect_pred"] = df_paired_val["control_itt_efficiency"]
df_paired_val["ate_pred"] = df_paired_val["control_itt_efficiency"] + ate_estimate

In [ ]:
best_prolific = df_predictions.groupby(["source","playerID"])["squared_error"].sum()["prolific",:].sort_values(ascending=True).index[0]
best_sspp = df_predictions.groupby(["source","playerID"])["squared_error"].sum()["sspp",:].sort_values(ascending=True).index[0]
median_prolific = df_predictions.groupby(["source","playerID"])["squared_error"].sum()["prolific",:].sort_values(ascending=True).index[int(np.floor(len(df_predictions.groupby(["source","playerID"])["squared_error"].sum()["prolific",:])/2))]
median_sspp = df_predictions.groupby(["source","playerID"])["squared_error"].sum()["sspp",:].sort_values(ascending=True).index[int(np.floor(len(df_predictions.groupby(["source","playerID"])["squared_error"].sum()["sspp",:])/2))]

In [ ]:
df_paired_val = (df_paired_val
 .merge(df_predictions.query("playerID == @best_prolific")[["CONFIG_configId", "prediction"]].rename(columns={"prediction":"best_prolific_pred"}), on="CONFIG_configId", how="left")
 .merge(df_predictions.query("playerID == @best_sspp")[["CONFIG_configId", "prediction"]].rename(columns={"prediction":"best_sspp_pred"}), on="CONFIG_configId", how="left")
 .merge(df_predictions.query("playerID == @median_prolific")[["CONFIG_configId", "prediction"]].rename(columns={"prediction":"median_prolific_pred"}), on="CONFIG_configId", how="left")
 .merge(df_predictions.query("playerID == @median_sspp")[["CONFIG_configId", "prediction"]].rename(columns={"prediction":"median_sspp_pred"}), on="CONFIG_configId", how="left")
 .merge(df_predictions.query("source == 'prolific'").groupby("CONFIG_configId")["prediction"].median().reset_index().rename(columns={"prediction":"woc_prolific_pred"}), on="CONFIG_configId", how="left")
 .merge(df_predictions.query("source == 'sspp'").groupby("CONFIG_configId")["prediction"].median().reset_index().rename(columns={"prediction":"woc_sspp_pred"}), on="CONFIG_configId", how="left")
)

# Figure 3 code

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.gridspec import GridSpec
import seaborn as sns
from matplotlib.ticker import MaxNLocator
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import pandas as pd
import numpy as np
from scipy.stats import pearsonr


# Define the labels mapping
dict_perf_fig_labels = {
    "median_prolific_pred": "Median\nlayperson",
    "median_sspp_pred": "Median\nexpert",
    "woc_sspp_pred": "WOC\nexperts",
    "best_sspp_pred": "Best\nexpert",
    "xgb_prereg_pred": "XGB",
    "best_prolific_pred": "Best\nlayperson",
    "woc_prolific_pred": "WOC\nlaypeople",
    "mlp_prereg_pred": "MLP",
    "rf_prereg_pred": "RF",
    "ols_prereg_pred": "OLS",
    "elastic_prereg_pred": "E-Net"
}



def predictive_scatterplot(axis, prediction_col, panel_label="", scatter_color='#0072B2'):
    # Define colors
    mean_line_color = '#666666'  # darker red
    identity_line_color = '#000000'  # dark gray
    fill_color = '#95D840'  # green
    
    # Plot mean line
    axis.axhline(100*df_paired_learn["treatment_itt_efficiency"].mean(), 
                 color=mean_line_color, 
                 linestyle="--", 
                 linewidth=1, 
                 label="Mean in the learning dataset",
                 zorder=1)
    
    # Plot identity line
    min_val = 100*df_paired_val["treatment_itt_efficiency"].min()
    max_val = 100*df_paired_val["treatment_itt_efficiency"].max()
    axis.plot([min_val, max_val], 
              [min_val, max_val], 
              linestyle="-", 
              color=identity_line_color, 
              linewidth=1, 
              label="Identity",
              zorder=2)
    
    # Add fill
    axis.fill_between(x=[min_val, max_val],
                      y1=[min_val, max_val],
                      y2=100*df_paired_learn["treatment_itt_efficiency"].mean(), 
                      color=fill_color, 
                      alpha=0.25,
                      zorder=0)
    
    # Modified scatter plot with filled circles and opacity
    axis.scatter(x=100*df_paired_val["treatment_itt_efficiency"], 
                y=100*df_paired_val[prediction_col], 
                facecolor=scatter_color,
                edgecolor='none',
                s=25,
                alpha=0.6,
                zorder=3)
    
    # Calculate statistics
    temp_r2 = 1 - np.sum((df_paired_val[prediction_col] - df_paired_val["treatment_itt_efficiency"])**2) / \
              np.sum((df_paired_val["baseline"] - df_paired_val["treatment_itt_efficiency"])**2)
    temp_pearsonr = pearsonr(df_paired_val[prediction_col], df_paired_val["treatment_itt_efficiency"]).statistic
    
    # Add statistics text with smaller font
    axis.text(0.55, 0.23, f'$R^2 = {temp_r2.round(2)}$', 
             transform=axis.transAxes, 
             fontsize=8, 
             fontweight='normal', 
             va='top', 
             ha='left')
    axis.text(0.55, 0.13, f'$r = {temp_pearsonr.round(2)}$', 
             transform=axis.transAxes, 
             fontsize=8, 
             fontweight='normal', 
             va='top', 
             ha='left')
    
    # Add panel label
    axis.text(0.05, 0.95, panel_label, 
             transform=axis.transAxes, 
             fontsize=11, 
             fontweight='bold', 
             va='top', 
             ha='left')
    
    # Set shared axis limits
    axis.set_xlim(60, 100)
    axis.set_ylim(60, 100)

# Main plotting code
plt.style.use('default')
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
matplotlib.rcParams['font.sans-serif'] = ['Arial']
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['axes.facecolor'] = 'white'
matplotlib.rcParams['figure.facecolor'] = 'white'
matplotlib.rcParams['grid.alpha'] = 0.0

# Create figure
width_mm = 183
height_mm = 200
width_inches = width_mm / 25.4
height_inches = height_mm / 25.4
fig = plt.figure(figsize=(width_inches, height_inches), dpi=300)

# Create GridSpec
gs = GridSpec(4, 3, width_ratios=[1.2, 1, 1], height_ratios=[1, 1, 1, 1],
             hspace=0.4, wspace=0.2)

# Initialize all axes
ax1 = fig.add_subplot(gs[:, 0])
ax2_1 = fig.add_subplot(gs[0, 1])
ax2_2 = fig.add_subplot(gs[0, 2])
ax2_3 = fig.add_subplot(gs[1, 1])
ax2_4 = fig.add_subplot(gs[1, 2])
ax2_5 = fig.add_subplot(gs[2, 1])
ax2_6 = fig.add_subplot(gs[2, 2])
ax2_7 = fig.add_subplot(gs[3, 1])
ax2_8 = fig.add_subplot(gs[3, 2])

ax = [ax2_1, ax2_2, ax2_3, ax2_4, ax2_5, ax2_6, ax2_7, ax2_8]

# Define group colors - darker and more saturated
group_colors = {
    'statistical': '#0072B2',  # Dark blue (kept)
    'experts': '#CCA97A',      # Dark orange (kept)
    'laypeople': '#E41A1C'     # Rich purple
}


# Create a color mapping for each model
color_mapping = {
    'elastic_prereg_pred': group_colors['statistical'],
    'ols_prereg_pred': group_colors['statistical'],
    'rf_prereg_pred': group_colors['statistical'],
    'mlp_prereg_pred': group_colors['statistical'],
    'xgb_prereg_pred': group_colors['statistical'],
    'best_sspp_pred': group_colors['experts'],
    'woc_sspp_pred': group_colors['experts'],
    'median_sspp_pred': group_colors['experts'],
    'best_prolific_pred': group_colors['laypeople'],
    'woc_prolific_pred': group_colors['laypeople'],
    'median_prolific_pred': group_colors['laypeople']
}

# First panel (vertical) with colored bars
performance_data = pd.DataFrame({
    'model': list(color_mapping.keys()),
    'performance': [np.sqrt(np.mean((df_paired_val[col]*100 - df_paired_val['treatment_itt_efficiency']*100)**2)) 
                   for col in color_mapping.keys()]
}).sort_values('performance', ascending=True)

bar_colors = [color_mapping[col] for col in performance_data['model']]

# Create the bar plot
sns.barplot(y="model", x="performance", 
           data=performance_data,
           palette=bar_colors,
           ax=ax1)

# Update y-tick labels with the proper names
ax1.set_yticks(range(len(performance_data)))
ax1.set_yticklabels([dict_perf_fig_labels[model] for model in performance_data['model']], 
                    fontsize=8)

# Style main bar plot
ax1.set_ylabel("", fontsize=10, labelpad=10)
ax1.set_xlabel("RMSE", fontsize=10, labelpad=10)
ax1.tick_params(axis='both', which='major', labelsize=8, length=4, width=0.5)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.spines['left'].set_linewidth(0.5)
ax1.spines['bottom'].set_linewidth(0.5)

# Add baseline
baseline_color = '#666666'  # darker red
ax1.axvline(x=np.sqrt(np.mean((df_paired_val['baseline']*100 - df_paired_val['treatment_itt_efficiency']*100)**2)), 
            linestyle='--', color=baseline_color, 
            label="Mean in the learning dataset",
            linewidth=1)

# Add panel letter "A"
ax1.text(0.05, 1.02, "A", fontsize=11, fontweight='bold', transform=ax1.transAxes)

# Create unified legend elements
legend_elements = [
    Patch(facecolor=group_colors['statistical'], label='Statistical models'),
    Patch(facecolor=group_colors['experts'], label='Experts'),
    Patch(facecolor=group_colors['laypeople'], label='Laypeople'),
    Line2D([0], [0], color=baseline_color, linestyle="--", label='Mean in the learning dataset'),
    Line2D([0], [0], color='#000000', linestyle='-', label='Identity')
]

# Add unified legend at the top of the figure
fig.legend(handles=legend_elements, 
          loc='upper center',
          bbox_to_anchor=(0.5, 0.98),
          ncol=5,
          frameon=False,
          fontsize=8,
          borderaxespad=0,
          handlelength=1.5)

# Calculate global min/max for shared axes
plot_order = ["elastic_prereg_pred", "ols_prereg_pred", "best_prolific_pred", 
              "best_sspp_pred", "woc_prolific_pred", "woc_sspp_pred", 
              "median_prolific_pred", "median_sspp_pred"]

all_vals = []
for col in plot_order:
    all_vals.extend(100*df_paired_val[col])
all_vals.extend(100*df_paired_val["treatment_itt_efficiency"])
global_min = min(all_vals)
global_max = max(all_vals)

# Add padding to limits
padding = (global_max - global_min) * 0.05
global_min -= padding
global_max += padding

# Style all scatter plots
for idx, axi in enumerate(ax):
    predictive_scatterplot(axi, plot_order[idx], 
                          ["B", "C", "D", "E", "F", "G", "H", "I"][idx],
                          scatter_color=color_mapping[plot_order[idx]])
    
    # Set titles and style
    titles = ["E-Net", "OLS", "Best layperson", "Best expert", 
              "Layperson wisdom of crowds", "Expert wisdom of crowds",
              "Median layperson", "Median expert"]
    axi.set_title(titles[idx], fontsize=9, pad=8)
    
    # Style scatter plots
    axi.tick_params(axis='both', which='major', labelsize=8, length=3, width=0.5)
    axi.spines['top'].set_visible(False)
    axi.spines['right'].set_visible(False)
    axi.spines['left'].set_linewidth(0.5)
    axi.spines['bottom'].set_linewidth(0.5)
    
    # Remove labels from inner axes
    if idx not in [6, 7]:
        axi.set_xlabel('')
    if idx % 2 != 0:
        axi.set_ylabel('')

# Adjust layout with updated spacing
plt.subplots_adjust(left=0.15, right=0.95, bottom=0.12, top=0.92, wspace=0.2)

# Add labels with adjusted positioning
scatter_center = 0.7
fig.supxlabel("True Efficiency", fontsize=10, x=scatter_center, y=0.064)
fig.text(0.4, 0.5, "Predicted Efficiency", fontsize=10, rotation=90, va='center')

# Save
plt.savefig('PGG-Fig3.pdf', dpi=300, bbox_inches='tight')
#plt.close()